In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt

df_mortalidad = pd.read_parquet("mortalidad_limpio.parquet")

df_mortalidad = df_mortalidad[df_mortalidad['ANO'] < 2026]

In [3]:
df_mortalidad['LOCALIDAD'] = (df_mortalidad['LOCALIDAD']
    .str.replace('¢', 'ó').str.replace('‚', 'é')
    .str.replace('¡', 'í').str.replace('¤', 'ñ')
    .str.replace(r'Engativ\s?', 'Engativá ', regex=True)
    .str.replace(r'M.rtires', 'Mártires', regex=True))
df_mortalidad.to_parquet('mortalidad_limpio.parquet', index=False)

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Segmenta muertes por localidad, contando registros por grupo
resumen = df_mortalidad.groupby('LOCALIDAD').size().reset_index(name='total_muertes')

# Excluye localidad agregada ("00") y sin información ("99")
resumen = resumen[~resumen['LOCALIDAD'].str.startswith(('00','99'))]

# Crea 3 grupos de riesgo por percentil según total de muertes
resumen['grupo'] = pd.qcut(resumen['total_muertes'], q=3, labels=['Bajo riesgo','Riesgo medio','Alto riesgo'])
print(resumen.sort_values('total_muertes', ascending=False))

# Agrupa muertes por año y causa (para la línea temporal del dashboard)
tendencia = df_mortalidad.groupby(['ANO','CLASIFICACION_CRONICAS']).size().reset_index(name='total')

# Ordena localidades de menor a mayor para el gráfico de barras horizontales
por_localidad = resumen.sort_values('total_muertes', ascending=True)

# Tabla cruzada régimen x causa (ya usada en el Día 6)
tabla_regimen = pd.crosstab(df_mortalidad['REGIMEN_SEGURIDAD_SOCIAL'], df_mortalidad['CLASIFICACION_CRONICAS'])

# Crea grilla 2x2: la primera gráfica ocupa las 2 columnas de arriba
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Tendencia por causa', 'Localidades por segmento de riesgo', 'Régimen x Causa', ''),
    specs=[[{"colspan":2}, None],[{}, {}]]
)

# Agrega una línea por cada causa a la posición (fila 1, col 1)
for causa in tendencia['CLASIFICACION_CRONICAS'].unique():
    sub = tendencia[tendencia['CLASIFICACION_CRONICAS']==causa]
    fig.add_trace(go.Scatter(x=sub['ANO'], y=sub['total'], mode='lines', name=causa), row=1, col=1)

# Agrega barras horizontales de localidades a (fila 2, col 1)
fig.add_trace(go.Bar(x=por_localidad['total_muertes'], y=por_localidad['LOCALIDAD'], orientation='h', showlegend=False), row=2, col=1)

# Agrega el heatmap régimen x causa a (fila 2, col 2)
fig.add_trace(go.Heatmap(z=tabla_regimen.values, x=tabla_regimen.columns, y=tabla_regimen.index, colorscale='YlOrRd', showscale=False), row=2, col=2)

# Ajusta tamaño y título general
fig.update_layout(height=900, width=1100, title_text="Dashboard Mortalidad Prematura Bogotá 2010-2025")

# Guarda como HTML interactivo
fig.write_html('08_dashboard.html')

# Muestra en el notebook
fig.show()


                  LOCALIDAD  total_muertes         grupo
8              08 - Kennedy          10385   Alto riesgo
11                11 - Suba          10094   Alto riesgo
10          10 - Engativá             8424   Alto riesgo
19      19 - Ciudad Bolívar           6955   Alto riesgo
7                 07 - Bosa           6768   Alto riesgo
1              01 - Usaquén           5008   Alto riesgo
4        04 - San Cristóbal           4911   Alto riesgo
18  18 - Rafael Uribe Uribe           4886  Riesgo medio
5                 05 - Usme           4050  Riesgo medio
9             09 - Fontibón           3504  Riesgo medio
16       16 - Puente Aranda           2948  Riesgo medio
6           06 - Tunjuelito           2226  Riesgo medio
3             03 - Santa Fe           1744  Riesgo medio
12      12 - Barrios Unidos           1690   Bajo riesgo
13         13 - Teusaquillo           1629   Bajo riesgo
2            02 - Chapinero           1527   Bajo riesgo
14        14 - Los Mártires    